# Lab Experiment 1
Reading tabular, textual, image, signal and medical data

## Setup

In [ ]:
!pip -q install ucimlrepo wfdb

In [ ]:
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from ucimlrepo import fetch_ucirepo
import wfdb

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

## Tabular and Medical Data

In [ ]:
heart_disease = fetch_ucirepo(id=45)

X = heart_disease.data.features.copy()
y_raw = heart_disease.data.targets.copy()

for column in X.columns:
    X[column] = pd.to_numeric(X[column], errors="coerce")

y_numeric = pd.to_numeric(y_raw.iloc[:, 0], errors="coerce")
y = (y_numeric > 0).astype(int)
y.name = "heart_disease"

heart_df = X.copy()
heart_df[y.name] = y

print("Dataset shape:", heart_df.shape)
print()
print("First 5 rows:")
display(heart_df.head())

In [ ]:
print("Column data types:")
print(heart_df.dtypes)

print()
print("Missing values per column:")
print(heart_df.isna().sum())

print()
print("Target distribution:")
print(heart_df["heart_disease"].value_counts())

### CSV and JSON

In [ ]:
sample_tabular = heart_df.head(10)

sample_tabular.to_csv("heart_sample.csv", index=False)
sample_tabular.to_json("heart_sample.json", orient="records", indent=2)

csv_data = pd.read_csv("heart_sample.csv")
json_data = pd.read_json("heart_sample.json")

print("CSV data shape:", csv_data.shape)
print("JSON data shape:", json_data.shape)

display(csv_data.head(3))

### Analysis

In [ ]:
correlations = (
    heart_df.corr(numeric_only=True)["heart_disease"]
    .drop("heart_disease")
    .sort_values(key=abs, ascending=False)
)

print(correlations)

plt.figure(figsize=(10, 5))
correlations.plot(kind="bar")
plt.title("Feature Correlation with Heart Disease Target")
plt.ylabel("Correlation")
plt.xlabel("Feature")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000)),
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print(f"Accuracy: {accuracy:.4f}")
print(f"ROC AUC:  {roc_auc:.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred, digits=4))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Disease", "Disease"]).plot()
plt.title("Confusion Matrix - Logistic Regression")
plt.show()

## Signal Data

In [ ]:
record_name = "100"
seconds_to_load = 10
samples_to_load = 360 * seconds_to_load

record = wfdb.rdrecord(
    record_name,
    sampfrom=0,
    sampto=samples_to_load,
    pn_dir="mitdb",
)

annotation = wfdb.rdann(
    record_name,
    "atr",
    sampfrom=0,
    sampto=samples_to_load,
    pn_dir="mitdb",
)

print("Sampling frequency:", record.fs, "Hz")
print("Signal names:", record.sig_name)
print("Signal shape:", record.p_signal.shape)
print("Number of annotations in segment:", len(annotation.sample))

In [ ]:
fs = record.fs
time = np.arange(record.p_signal.shape[0]) / fs
signal = record.p_signal[:, 0]

plt.figure(figsize=(14, 4))
plt.plot(time, signal, linewidth=1)

valid_ann = annotation.sample[annotation.sample < len(signal)]
plt.scatter(
    valid_ann / fs,
    signal[valid_ann],
    marker="x",
    label="Beat annotations",
)

plt.title("MIT-BIH Record 100 - ECG Segment")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude (mV)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
if len(annotation.sample) >= 2:
    rr_intervals = np.diff(annotation.sample) / fs
    heart_rates = 60 / rr_intervals

    print(f"Mean R-R interval: {rr_intervals.mean():.3f} s")
    print(f"Mean heart rate:   {heart_rates.mean():.2f} bpm")
    print(f"Minimum heart rate: {heart_rates.min():.2f} bpm")
    print(f"Maximum heart rate: {heart_rates.max():.2f} bpm")
else:
    print("Not enough beat annotations to calculate R-R intervals.")

## Text Data

In [ ]:
clinical_note = (
    "Patient reports intermittent chest pain during physical activity. "
    "Pain improves with rest. The patient also reports mild shortness of breath "
    "and fatigue. No fainting episode was reported."
)

with open("clinical_note.txt", "w", encoding="utf-8") as file:
    file.write(clinical_note)

pd.DataFrame([
    {"patient_id": "SYN001", "note": clinical_note}
]).to_json("clinical_note.json", orient="records", indent=2)

with open("clinical_note.txt", "r", encoding="utf-8") as file:
    text_from_txt = file.read()

text_from_json = pd.read_json("clinical_note.json").loc[0, "note"]

print("TXT note:")
print(text_from_txt)
print()
print("JSON note matches TXT:", text_from_json == text_from_txt)

In [ ]:
stop_words = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for", "from",
    "in", "is", "it", "of", "on", "or", "the", "to", "was", "with"
}

tokens = re.findall(r"[A-Za-z]+", text_from_txt.lower())
filtered_tokens = [word for word in tokens if word not in stop_words]
word_frequency = Counter(filtered_tokens)

print("Tokens:", filtered_tokens)
print()
print("Most common terms:")
for word, count in word_frequency.most_common(10):
    print(f"{word}: {count}")

## Image Data

In [ ]:
size = 256
y_grid, x_grid = np.ogrid[:size, :size]
distance = np.sqrt((x_grid - size / 2) ** 2 + (y_grid - size / 2) ** 2)
image_array = np.clip(230 - 1.2 * distance, 0, 255).astype(np.uint8)

synthetic_image = Image.fromarray(image_array, mode="L")
synthetic_image.save("medical_image.png")
synthetic_image.save("medical_image.jpg", quality=95)

png_image = Image.open("medical_image.png")
jpg_image = Image.open("medical_image.jpg")

print("PNG - format:", png_image.format, "size:", png_image.size, "mode:", png_image.mode)
print("JPEG - format:", jpg_image.format, "size:", jpg_image.size, "mode:", jpg_image.mode)

In [ ]:
resized_image = png_image.resize((128, 128))
normalized_pixels = np.asarray(resized_image, dtype=np.float32) / 255.0

print("Resized image shape:", normalized_pixels.shape)
print("Normalized pixel range:", normalized_pixels.min(), "to", normalized_pixels.max())

plt.figure(figsize=(5, 5))
plt.imshow(resized_image, cmap="gray")
plt.title("Loaded and Resized Grayscale Image")
plt.axis("off")
plt.show()

## Integration
The different data types can be linked using patient ID and time. ECG features, clinical values, images and notes can then be used together.

## Conclusion
Tabular, medical, signal, text and image data were read and processed using Python.

## References
- UCI Heart Disease Dataset
- PhysioNet MIT-BIH Arrhythmia Database